# 11 - Structured output and your own tools (take-home)

**ICSSI 2026, Using LLMs via the API**

Almost every project needs two things: reliable JSON out of the model, and the ability to
give the model your own tools.

In [ ]:
# fetch the repo files (claude_kit.py and tutorial_data/) when running in Colab
import os, sys, subprocess
REPO_URL = "https://github.com/LarremoreLab/icssi-2026-hackathon.git"  # update after you push to GitHub

if "google.colab" in sys.modules and not os.path.exists("claude_kit.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))

# this notebook lives in take_home/; walk up to the repo root so claude_kit.py,
# .env and tutorial_data/ all resolve no matter which folder the kernel started in
while not os.path.exists("claude_kit.py") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("working directory:", os.getcwd())
assert os.path.exists("claude_kit.py"), (
    "claude_kit.py was not found. Run this notebook from inside the cloned repo folder, "
    "or set REPO_URL above and run this cell again in Colab."
)

In [ ]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        try:
            from dotenv import load_dotenv
        except ImportError:
            pip_install("python-dotenv")
            from dotenv import load_dotenv
        load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")

## Reliable JSON with messages.parse

Define the shape you want as a Pydantic model and let the SDK validate the response into
it. Here we extract structured metadata from an abstract.

In [ ]:
import json, anthropic
from pydantic import BaseModel

client = anthropic.Anthropic()
abstracts = json.load(open("tutorial_data/abstracts.json"))

class AbstractMeta(BaseModel):
    title: str
    methods: list[str]
    datasets_used: list[str]
    main_contribution: str

abstract = abstracts[2]
response = client.messages.parse(
    model="claude-opus-4-8", max_tokens=500,
    messages=[{"role": "user",
               "content": "Extract metadata from this abstract:\n\n" + abstract["abstract"]}],
    output_format=AbstractMeta,
)
metadata = response.parsed_output            # a validated AbstractMeta instance
print(metadata.model_dump_json(indent=2))

## Custom tools let the model call your code

You describe a tool with a name, a description, and a JSON input schema. When Claude wants
it, the response has `stop_reason` equal to `tool_use`. You run the function, send back a
`tool_result`, and let Claude finish. This is the manual loop that powers agents.

In [ ]:
# a toy database of citation counts for our sample papers
citation_counts = {item["id"]: 100 + 37 * index for index, item in enumerate(abstracts)}

tools = [{
    "name": "lookup_citation_count",
    "description": "Return the citation count for a paper given its ID, for example P003.",
    "input_schema": {
        "type": "object",
        "properties": {"paper_id": {"type": "string", "description": "paper id like P001"}},
        "required": ["paper_id"],
    },
}]

def run_tool(name, arguments):
    if name == "lookup_citation_count":
        return str(citation_counts.get(arguments["paper_id"], "unknown"))
    return "unknown tool"

messages = [{"role": "user", "content": "How many citations does paper P003 have? Use the tool."}]
while True:
    response = client.messages.create(model="claude-opus-4-8", max_tokens=400,
                                      tools=tools, messages=messages)
    if response.stop_reason != "tool_use":
        break
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            print("[tool call]", block.name, block.input)
            tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                 "content": run_tool(block.name, block.input)})
    messages.append({"role": "user", "content": tool_results})

print("\nfinal answer:", response.content[0].text)

## Recap

Calling `messages.parse(..., output_format=Model)` gives you validated, typed JSON. A
custom tool is a name, a description, and an input schema. You execute it and return a
`tool_result`, looping until `stop_reason` is no longer `tool_use`. Swap the toy lookup for
a real call, such as OpenAlex, your database, or an internal API, and you have a domain
agent.